# Import dependencies


In [ ]:
import xarray as xr
import numpy as np
import torch

import matplotlib.pyplot as plt
import sys
from neuralop.models import FNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.utils import count_model_params
from neuralop import LpLoss, H1Loss

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [ ]:
# Each sample shape: [channels, lat, lon]
# Channels:
# 0: SSH
# 1: U velocity
# 2: V velocity
# 3: Wind U
# 4: Wind V
# 5: Sea level pressure
# 6: Bathymetry (static)

# Input shape:  [batch, 7, lat, lon]
# Output shape: [batch, 3, lat, lon]  (SSH, U, V at t+1)

# Loading the dataset
We load the Nordic sea dataset we have been provided. The dataset contains velocity and height (NEMO) in addition to forcing variables sea level pressure and winds (ERA5). In addition we have static variables bathymetry.

## Initialising the FNO model


In [ ]:
model = FNO(
    n_modes=(16, 32),
    in_channels=3,
    out_channels=3,
    hidden_channels=64,
    domain_padding=[0.05, 0.05],
    n_layers=2,
)
model = model.to(device)

# Count and display the number of parameters
n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()

# Initialising optimiser and scheduler

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# Setting up loss functions

In [ ]:
l2loss = LpLoss(d=2, p=2, reduction="sum")

train_loss = l2loss
eval_losses = {"l2": l2loss}

In [ ]:
print("\n### MODEL ###\n", model)
print("\n### OPTIMIZER ###\n", optimizer)
print("\n### SCHEDULER ###\n", scheduler)
print("\n### LOSSES ###")
print(f"\n * Train: {train_loss}")
print(f"\n * Test: {eval_losses}")
sys.stdout.flush()

# Creating trainer

In [ ]:
trainer = Trainer(
    model=model,
    n_epochs=30,
    device=device,
    wandb_log=False,  # Disable Weights & Biases logging
    eval_interval=5,  # Evaluate every 5 epochs
    use_distributed=False,  # Single GPU/CPU training
    verbose=True,  # Print training progress
)

# Training model

In [ ]:
trainer.train(
    train_loader=train_loader,
    test_loaders=test_loaders,
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    eval_losses=eval_losses,
)

Training loop concept
# Preprocessing the dataset
## Normalise the variables

In [2]:
# Example for SSH
ssh_mean = ssh.mean()
ssh_std = ssh.std()
ssh_normalized = (ssh - ssh_mean) / ssh_std

NameError: name 'ssh' is not defined

In [ ]:
# For each timestep t:
input_t = torch.stack([
    ssh[t], u[t], v[t],        # current state
    wind_u[t], wind_v[t],      # forcings
    slp[t],                     # forcing
    bathymetry                  # static
], dim=0)

target_t = torch.stack([
    ssh[t+1], u[t+1], v[t+1]  # next state to predict
], dim=0)

In [ ]:
tau_x = rho_air * Cd * sqrt(u10**2 + v10**2) * u10
tau_y = rho_air * Cd * sqrt(u10**2 + v10**2) * v10

In [ ]:
eta_ib = - msl / (rho_water * g)

In [ ]:
# # Load trained model
# fno_model = FNO(n_modes=(32, 32), hidden_channels=64, 
#                 in_channels=6, out_channels=3, n_layers=4).to(device)
# fno_model.load_state_dict(torch.load('fno_model.pt'))
# fno_model.eval()

# # Single step prediction
# with torch.no_grad():
#     sample_input = inputs_torch[[0]].to(device)  # Shape: [1, 6, x, y]
#     prediction = fno_model(sample_input)  # Shape: [1, 3, x, y]
    
#     pred_ssh = prediction[0, 0].cpu().numpy()
#     pred_ubar = prediction[0, 1].cpu().numpy()
#     pred_vbar = prediction[0, 2].cpu().numpy()

# # Multi-step prediction (autoregressive)
# current_state = inputs_torch[[0]].clone().to(device)
# predictions = []

# for t in range(24):  # Predict 24 steps ahead
#     with torch.no_grad():
#         next_pred = fno_model(current_state)  # [1, 3, x, y]
#     predictions.append(next_pred.cpu().numpy())
    
#     # Update state for next iteration
#     next_forcing = inputs_torch[[1]][0, :3, :, :]  # Next u10, v10, msl
#     next_state = torch.cat([
#         next_forcing.unsqueeze(0),
#         next_pred  # Use predicted ssh, ubar, vbar
#     ], dim=1)
#     current_state = next_state.to(device)

# predictions = np.array(predictions)  # Shape: [24, 1, 3, x, y]

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# axes[0].imshow(pred_ssh, cmap='viridis')
# axes[0].set_title('Predicted SSH')
# axes[0].colorbar()

# axes[1].imshow(pred_ubar, cmap='RdBu_r')
# axes[1].set_title('Predicted U-velocity')
# axes[1].colorbar()

# axes[2].imshow(pred_vbar, cmap='RdBu_r')
# axes[2].set_title('Predicted V-velocity')
# axes[2].colorbar()

# plt.tight_layout()
# plt.show()